# Phase 3 — reasoning finetuning, evaluation, and attention analysis

Both models, end to end, in one Colab session. Finetunes Model H (Hindi) and
Model L (Nepali) on synthetic comparative-reasoning corpora, scores each against
its own pretrained checkpoint, compares attention before and after, and writes
the report's comparison tables.

**Why one notebook.** Phase 2 split the two models across Colab and Kaggle
because Nepali pretraining alone was 4.87 hours of T4 and the two runs would not
fit on one free tier. Phase 3's whole GPU cost is under an hour, so the split
bought nothing and cost an rclone dependency, a Kaggle secret, and artifacts
landing in two places. Everything is on Drive now.

**Runtime: ~75 minutes**, or ~61 without the optional second seed in §7.
Six finetunes at ~6 min, eleven evaluations, four perplexity passes, two
attention passes. Corpus generation and the test suite need no GPU.

**Every cell is resumable.** Corpus generation is deterministic from its seed,
finetuning picks up from `finetune_latest.pt`, and evaluation just re-reads
checkpoints. If Colab disconnects, re-run from the setup cells down.

> All heavy work is a call into `src/`, not code pasted into this notebook —
> the same arrangement as Phase 2. The only inline Python here is mounting
> Drive and the pre-flight assertions.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
import os, shutil, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase3'

# Always take a fresh copy: Drive FUSE is slow for many small reads and cannot
# host __pycache__ writes cleanly, while /content/ is local NVMe.
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}

In [ ]:
!apt-get install -y -qq fonts-indic 2>&1 | tail -2
!pip install -q pyyaml sentencepiece

### Confirm the Drive code snapshot is the Phase 3 one

`{DRIVE_BASE}/code` is rclone-synced from the local repo. If it still holds the
Phase 2 snapshot, everything below fails at the first `src.reasoning` import —
so check now rather than three cells in.

In [ ]:
required = ['src/reasoning/generate.py', 'src/train/finetune.py',
            'src/eval/reasoning_eval.py', 'src/eval/attention_reasoning.py',
            'src/eval/compare_reasoning.py',
            'hindi/configs/finetune.yaml', 'nepali/configs/finetune.yaml']
missing = [p for p in required if not os.path.exists(p)]
assert not missing, (
    f'Drive code snapshot is stale — missing {missing}. Re-sync from the local '
    f'repo:  rclone copy . gdrive:24KaratAu/code --exclude "__pycache__/**" '
    f'--exclude ".git/**" --exclude "data/**"')
print('Phase 3 code present.')

## 2. Build the reasoning corpora

Generated here rather than downloaded, so the leakage checks run on the exact
bytes that get trained on. The Hindi and Nepali inventories in
`src/reasoning/entities.py` are written independently — the two corpora share
the generator and nothing else.

In [ ]:
!python -u -m src.reasoning.generate --language hindi \
    --out-dir data/hindi/reasoning \
    --train 40000 --val 5000 --test 5000 --test-seen 5000 --seed 42

In [ ]:
!python -u -m src.reasoning.generate --language nepali \
    --out-dir data/nepali/reasoning \
    --train 40000 --val 5000 --test 5000 --test-seen 5000 --seed 42

### Verify the corpora before spending GPU time

Two separate things are checked. The generator's own leakage report confirms no
entity or prompt straddles a split. The test suite goes further: it re-parses
the rendered Devanagari back into an ordering and checks every answer against
it. Reading the generator's metadata would only prove it agrees with itself — a
premise renderer that stated an inequality backwards would pass that and
silently invert part of the corpus.

In [ ]:
for lang in ('hindi', 'nepali'):
    s = json.load(open(f'data/{lang}/reasoning/stats.json'))
    lk = s['leakage']
    assert not lk['train_test_shared_entities'], lk
    assert lk['train_test_shared_prompts'] == 0, lk
    assert lk['val_test_shared_prompts'] == 0, lk
    assert lk['train_val_shared_prompts'] == 0, lk
    # The control split shares the train distribution on purpose, so its
    # entities overlap by design; what must not overlap is the prompts.
    assert lk['test_seen_shared_prompts_with_train'] == 0, lk
    assert lk['test_seen_shared_prompts_with_test'] == 0, lk
    assert not lk['test_seen_entities_outside_train_pool'], lk
    t = s['tokenizer']['train']
    print(f"{lang:7} leakage clean | test held-out phrasing "
          f"{s['splits']['test']['held_out_phrasing']:,}/5,000 | "
          f"test_seen {s['splits']['test_seen']['n']:,} | "
          f"{t['mean_prompt_tokens']} prompt tok | "
          f"{t['mean_numeral_tokens']} tok/quantity")

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# Archive to Drive: ~46 MB per language is too much for git, and the brief wants
# large artifacts on Drive with links in the README.
for lang in ('hindi', 'nepali'):
    !mkdir -p {DRIVE_BASE}/{lang}/reasoning
    !cp data/{lang}/reasoning/* {DRIVE_BASE}/{lang}/reasoning/
!du -sh {DRIVE_BASE}/hindi/reasoning {DRIVE_BASE}/nepali/reasoning

## 3. Finetune — four runs, ~6 min each

Each model starts from **its own language's** pretrained checkpoint, weights
only: the Phase 2 optimizer moments are deliberately discarded, because that run
ended at a 3e-5 learning rate and restoring its second moments would damp
exactly the updates this phase depends on.

Two answer styles per language. `direct` supervises `उत्तर: <answer>` (~3.4
tokens); `rationale` supervises the restated chain first (~19 tokens). Exact
match is scored on the final answer line either way, so the two are comparable
and only the supervision differs.

In [ ]:
import yaml
CONFIGS = ['hindi/configs/finetune.yaml', 'hindi/configs/finetune_rationale.yaml',
           'nepali/configs/finetune.yaml', 'nepali/configs/finetune_rationale.yaml']
for c in CONFIGS:
    cfg = yaml.safe_load(open(c))
    ok = os.path.exists(cfg['init_from'])
    print(f"{'OK ' if ok else 'MISSING'} {c:42} <- {cfg['init_from']}")
    assert ok, f"Phase 2 checkpoint not on Drive: {cfg['init_from']}"
    assert cfg['checkpoint']['mirror_to'] == '', 'Colab writes straight to Drive'
print('\nAll four runs ready.')

In [ ]:
!python -u -m src.train.finetune --config hindi/configs/finetune.yaml

In [ ]:
!python -u -m src.train.finetune --config hindi/configs/finetune_rationale.yaml

In [ ]:
!python -u -m src.train.finetune --config nepali/configs/finetune.yaml

In [ ]:
!python -u -m src.train.finetune --config nepali/configs/finetune_rationale.yaml

## 4. Exact match — pretrained vs finetuned

Three setups per model. **Zero-shot** shows the pretrained model has no notion
of the answer format; its format rate, not just its accuracy, is the finding.
**Few-shot (k=4)** supplies the format, so whatever gap remains is about
reasoning rather than formatting — demonstrations come from the *train* split,
so the test set's held-out entity pool is never shown. **Finetuned** is
zero-shot again, since the format is now learned.

In [ ]:
H, N = f'{DRIVE_BASE}/hindi', f'{DRIVE_BASE}/nepali'
PRE_H, PRE_N = f'{H}/checkpoints/checkpoint_best.pt', f'{N}/checkpoints/checkpoint_best.pt'
FTH_D, FTH_R = f'{H}/checkpoints_ft_direct/finetune_best.pt', f'{H}/checkpoints_ft_rationale/finetune_best.pt'
FTN_D, FTN_R = f'{N}/checkpoints_ft_direct/finetune_best.pt', f'{N}/checkpoints_ft_rationale/finetune_best.pt'

COMMON_H = ('--model-config hindi/configs/model.yaml '
            '--tokenizer tokenizers/hindi/hindi_bpe_8k.model '
            '--test data/hindi/reasoning/test.jsonl --language hindi')
COMMON_N = ('--model-config nepali/configs/model.yaml '
            '--tokenizer tokenizers/nepali/nepali_bpe_8k.model '
            '--test data/nepali/reasoning/test.jsonl --language nepali')
os.makedirs('hindi/eval', exist_ok=True)
os.makedirs('nepali/eval', exist_ok=True)

In [ ]:
# --- Model H ---------------------------------------------------------------
!python -u -m src.eval.reasoning_eval --checkpoint {PRE_H} {COMMON_H} \
    --mode zeroshot --answer-style direct \
    --out hindi/eval/reasoning_zeroshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {PRE_H} {COMMON_H} \
    --mode fewshot --shots 4 --train data/hindi/reasoning/train.jsonl \
    --answer-style direct --out hindi/eval/reasoning_fewshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FTH_D} {COMMON_H} \
    --mode finetuned --answer-style direct \
    --out hindi/eval/reasoning_finetuned_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FTH_R} {COMMON_H} \
    --mode finetuned --answer-style rationale \
    --out hindi/eval/reasoning_finetuned_rationale.json

In [ ]:
# --- Model L ---------------------------------------------------------------
!python -u -m src.eval.reasoning_eval --checkpoint {PRE_N} {COMMON_N} \
    --mode zeroshot --answer-style direct \
    --out nepali/eval/reasoning_zeroshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {PRE_N} {COMMON_N} \
    --mode fewshot --shots 4 --train data/nepali/reasoning/train.jsonl \
    --answer-style direct --out nepali/eval/reasoning_fewshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FTN_D} {COMMON_N} \
    --mode finetuned --answer-style direct \
    --out nepali/eval/reasoning_finetuned_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FTN_R} {COMMON_N} \
    --mode finetuned --answer-style rationale \
    --out nepali/eval/reasoning_finetuned_rationale.json

### The control split: seen entities, seen wording

The third of the three test sets. `test_seen` is distributionally identical to
training — train entity pool, shared realisations only — and was never used to
select a checkpoint, so it answers "did training fit the distribution at all"
without the optimism that scoring on `val` would carry.

Read it against the other two. High here and at chance on `test` means the model
memorised entity names rather than learning the relation. The three degrade
independently, which is why one pooled number cannot stand in for them.

In [ ]:
!python -u -m src.eval.reasoning_eval --checkpoint {FTH_D} \
    --model-config hindi/configs/model.yaml \
    --tokenizer tokenizers/hindi/hindi_bpe_8k.model \
    --test data/hindi/reasoning/test_seen.jsonl --language hindi \
    --mode finetuned --answer-style direct \
    --out hindi/eval/reasoning_finetuned_direct_seen.json

!python -u -m src.eval.reasoning_eval --checkpoint {FTN_D} \
    --model-config nepali/configs/model.yaml \
    --tokenizer tokenizers/nepali/nepali_bpe_8k.model \
    --test data/nepali/reasoning/test_seen.jsonl --language nepali \
    --mode finetuned --answer-style direct \
    --out nepali/eval/reasoning_finetuned_direct_seen.json

## 5. Did finetuning damage the language model?

Supervised finetuning on a narrow task can wreck general next-token prediction —
catastrophic forgetting. It is not a hypothetical: a worked example in the course
material goes from perplexity **1.33 to 194.7** after SFT.

So the same `src/eval/intrinsic.py` that produced the Phase 2 headline numbers is
pointed at the pretrained and finetuned checkpoints in turn, over the *pretraining*
validation split — ordinary Hindi and Nepali text, nothing to do with reasoning.
The comparison is the point; the absolute value is already known from Phase 2.

A 2M-token slice via `--max-tokens`, not the full 26M split. The distinction
between "unchanged" and "destroyed" does not need twelve minutes per checkpoint
to establish, and the flag records itself in the output so a partial score can
never be mistaken for a full one.

In [ ]:
# Four runs, written out rather than looped: `!` magics with line
# continuations inside an indented block are unreliable in IPython.
!python -u -m src.eval.intrinsic --checkpoint {PRE_H} \
    --model-config hindi/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/hindi/tokens --language hindi \
    --splits val --max-tokens 2000000 \
    --out hindi/eval/intrinsic_pretrained_ft.json

!python -u -m src.eval.intrinsic --checkpoint {FTH_D} \
    --model-config hindi/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/hindi/tokens --language hindi \
    --splits val --max-tokens 2000000 \
    --out hindi/eval/intrinsic_finetuned_ft.json

!python -u -m src.eval.intrinsic --checkpoint {PRE_N} \
    --model-config nepali/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/nepali/tokens --language nepali \
    --splits val --max-tokens 2000000 \
    --out nepali/eval/intrinsic_pretrained_ft.json

!python -u -m src.eval.intrinsic --checkpoint {FTN_D} \
    --model-config nepali/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/nepali/tokens --language nepali \
    --splits val --max-tokens 2000000 \
    --out nepali/eval/intrinsic_finetuned_ft.json

In [ ]:
print(f"{'model':8} {'perplexity before':>18} {'after':>10} {'change':>10}")
for lang in ('hindi', 'nepali'):
    a = json.load(open(f'{lang}/eval/intrinsic_pretrained_ft.json'))['splits']['val']
    b = json.load(open(f'{lang}/eval/intrinsic_finetuned_ft.json'))['splits']['val']
    pa, pb = a['perplexity'], b['perplexity']
    print(f"{lang:8} {pa:18.2f} {pb:10.2f} {(pb / pa - 1) * 100:+9.1f}%")
    assert not a['full_split'], 'token budget should be recorded'

## 6. Attention — pretrained vs finetuned

Both checkpoints run over the **same** reasoning prompts, batched without
padding: these models use learned *absolute* positions, so left-padding a batch
would shift every position index and silently change the input. Prompts are
bucketed by exact token length instead, so nothing is padded at all.

Heatmaps cover layer 0 and layer 7 — one early, one late, as the brief requires.

In [ ]:
!python -u -m src.eval.attention_reasoning \
    --pretrained {PRE_H} --finetuned {FTH_D} \
    --model-config hindi/configs/model.yaml \
    --tokenizer tokenizers/hindi/hindi_bpe_8k.model \
    --test data/hindi/reasoning/test.jsonl --language hindi --tag direct \
    --outdir report/figures --out hindi/eval/attention_reasoning.json

In [ ]:
!python -u -m src.eval.attention_reasoning \
    --pretrained {PRE_N} --finetuned {FTN_D} \
    --model-config nepali/configs/model.yaml \
    --tokenizer tokenizers/nepali/nepali_bpe_8k.model \
    --test data/nepali/reasoning/test.jsonl --language nepali --tag direct \
    --outdir report/figures --out nepali/eval/attention_reasoning.json

## 7. A second seed *(optional — ~14 min)*

Every number above is one run at one seed. The Phase 3 brief asks for more than
one, and without a second run the per-family differences later have no scale to
be judged against: a 3-point gap means nothing if reseeding moves the headline by
5.

Two extra runs, `direct` style only, seed 7 — enough to report a range on the
headline number. **Skip the two cells below and everything still works**;
`compare_reasoning` prints "one seed per configuration" instead of a spread, and
the report keeps it as a stated limitation.

`--override` redirects the checkpoint directory and log as well as the seed, so
the seed-42 runs above are not overwritten.

In [ ]:
!python -u -m src.train.finetune --config hindi/configs/finetune.yaml \
    --override seed=7 \
      checkpoint.dir={DRIVE_BASE}/hindi/checkpoints_ft_direct_s7 \
      log.jsonl={DRIVE_BASE}/hindi/logs/finetune_direct_s7.jsonl

!python -u -m src.train.finetune --config nepali/configs/finetune.yaml \
    --override seed=7 \
      checkpoint.dir={DRIVE_BASE}/nepali/checkpoints_ft_direct_s7 \
      log.jsonl={DRIVE_BASE}/nepali/logs/finetune_direct_s7.jsonl

In [ ]:
S7H = f'{DRIVE_BASE}/hindi/checkpoints_ft_direct_s7/finetune_best.pt'
S7N = f'{DRIVE_BASE}/nepali/checkpoints_ft_direct_s7/finetune_best.pt'

!python -u -m src.eval.reasoning_eval --checkpoint {S7H} {COMMON_H} \
    --mode finetuned --answer-style direct --seed 7 \
    --out hindi/eval/reasoning_finetuned_direct_s7.json

!python -u -m src.eval.reasoning_eval --checkpoint {S7N} {COMMON_N} \
    --mode finetuned --answer-style direct --seed 7 \
    --out nepali/eval/reasoning_finetuned_direct_s7.json

## 8. A gentler recipe *(~12 min)*

The runs above used 3 epochs at a peak learning rate of 1e-4, and §5 showed what
that cost: validation perplexity on ordinary text rose 55x for Model H and 118x
for Model L. The models became very good at emitting the answer template and
stopped being able to model their own language.

This section trains the same thing far more gently — **1 epoch at 3e-5**, a
third the learning rate and a third the steps — so the report can show a
trade-off rather than a single broken point. The question it answers: *how much
reasoning accuracy do you give up to keep the language model intact?*

Everything else is held fixed. Only `seed`, `optim.lr`, `train.epochs` and the
output paths change, via `--override`, so nothing above is overwritten.

In [ ]:
# Written out, not looped: `!` with a line continuation inside an indented
# block is unreliable in IPython.
!python -u -m src.train.finetune --config hindi/configs/finetune.yaml \
    --override optim.lr=3.0e-5 schedule.min_lr=3.0e-6 train.epochs=1 \
      checkpoint.dir={DRIVE_BASE}/hindi/checkpoints_ft_gentle \
      log.jsonl={DRIVE_BASE}/hindi/logs/finetune_gentle.jsonl

!python -u -m src.train.finetune --config nepali/configs/finetune.yaml \
    --override optim.lr=3.0e-5 schedule.min_lr=3.0e-6 train.epochs=1 \
      checkpoint.dir={DRIVE_BASE}/nepali/checkpoints_ft_gentle \
      log.jsonl={DRIVE_BASE}/nepali/logs/finetune_gentle.jsonl

In [ ]:
# Reasoning accuracy, and the control split, for the gentle runs.
GH = f'{DRIVE_BASE}/hindi/checkpoints_ft_gentle/finetune_best.pt'
GN = f'{DRIVE_BASE}/nepali/checkpoints_ft_gentle/finetune_best.pt'

!python -u -m src.eval.reasoning_eval --checkpoint {GH} {COMMON_H} \
    --mode finetuned --answer-style direct \
    --out hindi/eval/reasoning_gentle_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {GN} {COMMON_N} \
    --mode finetuned --answer-style direct \
    --out nepali/eval/reasoning_gentle_direct.json

In [ ]:
# The other half of the trade-off: did the language model survive this time?
!python -u -m src.eval.intrinsic --checkpoint {GH} \
    --model-config hindi/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/hindi/tokens --language hindi \
    --splits val --max-tokens 2000000 \
    --out hindi/eval/intrinsic_gentle_ft.json

!python -u -m src.eval.intrinsic --checkpoint {GN} \
    --model-config nepali/configs/model.yaml \
    --tokens-dir {DRIVE_BASE}/nepali/tokens --language nepali \
    --splits val --max-tokens 2000000 \
    --out nepali/eval/intrinsic_gentle_ft.json

In [ ]:
# The trade-off, both recipes side by side.
print(f"{'model':7} {'recipe':22} {'reasoning EM':>13} {'val PPL':>10} {'vs pretrained':>14}")
for lang in ('hindi', 'nepali'):
    base = json.load(open(f'{lang}/eval/intrinsic_pretrained_ft.json'))['splits']['val']['perplexity']
    for tag, em_f, px_f in (
            ('3 epochs @ 1e-4', f'{lang}/eval/reasoning_finetuned_direct.json',
             f'{lang}/eval/intrinsic_finetuned_ft.json'),
            ('1 epoch @ 3e-5', f'{lang}/eval/reasoning_gentle_direct.json',
             f'{lang}/eval/intrinsic_gentle_ft.json')):
        if not (os.path.exists(em_f) and os.path.exists(px_f)):
            print(f'  {lang:7} {tag:22} (not run)'); continue
        em = json.load(open(em_f))['exact_match']
        px = json.load(open(px_f))['splits']['val']['perplexity']
        print(f'  {lang:7} {tag:22} {em:13.4f} {px:10.2f} {px / base:13.1f}x')
    print(f'  {lang:7} {"pretrained (no finetune)":22} {0.0:13.4f} {base:10.2f} {1.0:13.1f}x')

## 9. Comparison tables and error analysis

`compare_reasoning` reads each result's own `setup` block rather than taking
labels on the command line, so a run cannot be mislabelled in the table.

In [ ]:
!python -u -m src.eval.compare_reasoning \
    --inputs hindi/eval/reasoning_*.json nepali/eval/reasoning_*.json \
    --out report/figures/reasoning_comparison.json \
    --md  report/figures/reasoning_comparison.md \
    --figure report/figures/reasoning_accuracy.png

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('report/figures/reasoning_comparison.md').read()))
display(Image('report/figures/reasoning_accuracy.png'))

In [ ]:
# Layer-level attention view — the part the report discusses. Guarded, so a
# partial run reports what is missing instead of raising.
for lang in ('hindi', 'nepali'):
    path = f'{lang}/eval/attention_reasoning.json'
    if not os.path.exists(path):
        print(f'{lang}: {path} not found — re-run §6'); continue
    d = json.load(open(path))
    c = d['comparison']
    print(f"\n=== {lang} ===  {c['roles_changed']}/{c['heads_total']} heads changed role")
    print('layer   distance pre -> ft      entropy_norm pre -> ft    prev-token pre -> ft')
    for lay in c['per_layer']:
        dd, ee, pp = lay['mean_distance'], lay['entropy_normalised'], lay['prev_token_mass']
        print(f"{lay['layer']:>5}   {dd['pretrained']:>7.3f} -> {dd['finetuned']:<7.3f}"
              f"   {ee['pretrained']:>7.3f} -> {ee['finetuned']:<7.3f}"
              f"   {pp['pretrained']:>7.3f} -> {pp['finetuned']:<7.3f}")
    print('role transitions:', json.dumps(c['role_transitions']))

## 10. Copy results back to Drive

The finetuned checkpoints and logs are already on Drive — the configs write
there directly. This pushes the evaluation JSONs and figures so they survive the
session, then lists what to pull into the repo and commit.

In [ ]:
# Per-language subfolders. Both languages use identical basenames
# (reasoning_finetuned_direct.json in each), so copying them into one flat
# directory silently destroys one set -- which is exactly what happened on the
# first run.
for lang in ('hindi', 'nepali'):
    !mkdir -p {DRIVE_BASE}/phase3/eval/{lang}
    !cp {lang}/eval/reasoning_*.json {DRIVE_BASE}/phase3/eval/{lang}/
    !cp {lang}/eval/intrinsic_*_ft.json {DRIVE_BASE}/phase3/eval/{lang}/
    !cp {lang}/eval/attention_reasoning.json {DRIVE_BASE}/phase3/eval/{lang}/

!mkdir -p {DRIVE_BASE}/phase3/figures
!cp report/figures/reasoning_* {DRIVE_BASE}/phase3/figures/
!cp report/figures/*_reasoning_attention_* {DRIVE_BASE}/phase3/figures/

# Verify both languages survived rather than assuming it.
for lang in ('hindi', 'nepali'):
    n = len(os.listdir(f'{DRIVE_BASE}/phase3/eval/{lang}'))
    print(f'  {lang}: {n} files on Drive')
    assert n >= 7, f'{lang} archive incomplete'

In [ ]:
print("""Pull these into the local repo and commit:

  hindi/eval/reasoning_*.json            nepali/eval/reasoning_*.json
    (zeroshot, fewshot, finetuned x2, plus *_seen and *_s7 if run)
  hindi/eval/intrinsic_*_ft.json         nepali/eval/intrinsic_*_ft.json
  hindi/eval/attention_reasoning.json    nepali/eval/attention_reasoning.json
  hindi/logs/finetune_*.jsonl            nepali/logs/finetune_*.jsonl
  report/figures/reasoning_comparison.{md,json}
  report/figures/reasoning_accuracy.png
  report/figures/*_reasoning_attention_direct_layer{0,7}.png
  report/figures/*_reasoning_attention_direct_prompt.txt
  data/*/reasoning/{stats.json,sample.md}   <- regenerated, now include test_seen

Finetuned checkpoints stay on Drive (304.7 MB each); README links already point
at them. Then the report's [pending run] sections can be filled: SS5.1-5.5, SS6,
and the completions in SS7.2-7.4 and SS10.""")